# 19.12 完成整合後，怎樣回答「這個模型會什麼」？


回到最初的問題：同一個自行訓練的核心，能否讀文字、接圖片與錄音、使用工具，再保留對話要求？答案要按任務說明。模型先由[驗證資料選定](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/results/v2-final-training-source-selection.json)，固定推論權重與評分規則後，MoE與Dense各完成一次3734題的最後測試；這份最後考卷沒有再拿來訓練或挑版本。

下表集中列出主要能力。分母是**被評分的問題數**，不是互相獨立的原圖、錄音或人數。文字與客服題依固定關鍵詞及內容規則檢查語意；OCR、服飾與位置題檢查要求的答案；「真正往返」還要通過表中整條鏈。

| 最後任務 | 這一列要求什麼 | MoE | Dense |
| --- | --- | ---: | ---: |
| 有限文字與指令 | 回應範圍內問題，依歷史保留條件與必要內容。 | 464／560 | 429／560 |
| 指定區域OCR | 讀出所給文字框內的完整字串。 | 252／324 | 281／324 |
| 三類服飾 | 回答指定商品的類別。 | 357／360 | 356／360 |
| 兩格位置關係 | 回答左／右／上／下指定格位的商品類別。 | 1427／1440 | 1426／1440 |
| 語音客服回答 | 從錄音回答地址、App或卡片問題。 | 42／90 | 37／90 |
| 真正語音後續對話 | 第一輪回答正確，再用模型自己生成的歷史完成下一輪要求。 | 26／60 | 24／60 |
| 真正計算器往返 | 產生正確工具與參數、真正執行，再讀回結果答對。 | 0／276 | 7／276 |
| 已提供工具返回值 | 已給定工具訊息後，依返回值生成回答。 | 551／552 | 468／552 |

最後兩列回答不同問題。已提供返回值的題目省去了「模型自己提出正確請求」，所以高分能支持讀回結果，不能替真正往返補分。語音後續對話也沒有貼入標準第一輪答案；第一輪答錯，即使下一輪把錯話改成兩點格式，整條鏈仍算錯。

再回收[19.3](https://birdhackor.github.io/tiny-perceptron-vlm/19.3.html)的答案分佈檢查：360道單物件服飾題中，褲子、包、短靴各120題。完全不看圖、固定答「這是包。」只能對120／360，也就是三分之一；這是按標準答案分佈算出的基線，不是另跑一次模型。上表的357／360與356／360超過這個固定猜法，接著還要看保持問題、改變材料或歷史時，答案是否跟著改對。

下表整理同一份最後考卷中的控制組。交換位置與換讀字框，每組2題；主題控制每組3題，其他歷史控制每組2題。後兩欄依序列出MoE、Dense的組數，分母寫在第一欄。「全組皆正確」要求組內每題依本節判準檢查內容都正確，且有格式要求時也通過；「回答有改變」只檢查按既有規則整理後的輸出至少有兩種，不等於改對。

| 控制與答案預期（分母：組） | 全組皆正確：MoE、Dense | 回答有改變：MoE、Dense |
| --- | ---: | ---: |
| 交換商品位置（720組）：固定位置問句與兩件商品，交換實際像素的位置；應答出移到指定位置的商品。 | 707、706 | 715、714 |
| 同圖換讀字框（150組）：固定字卡，換框與框內字串；應只抄新框內的完整文字。 | 86、110 | 150、150 |
| 改歷史主題（40組）：固定這輪改寫要求，前文分別談地址、App或卡片；應改寫各自的主題。 | 31、33 | 40、40 |
| 改歷史格式（140組）：固定這輪問題，前文改成要求一句或兩點；應保留正確內容並採指定格式。 | 103、113 | 138、140 |
| 改歷史選項（80組）：固定這輪詢問與格式，改前文最後選定的物件；應回答最新選項。 | 63、0 | 66、69 |

例如換讀字框時，兩版150組的回答都改了，但只有86組與110組的兩題都抄對；Dense的選項控制有69組改答，卻沒有一組兩題都答對。這些觀察能核對[19.6](https://birdhackor.github.io/tiny-perceptron-vlm/19.6.html)承諾的材料、範圍與歷史依賴，也保留了依賴失敗的位置。它們是上述考卷題目的重新分組，不是額外的獨立樣本；條件與計數可回查[資料準備](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/scripts/selftrained/prepare_vision_ocr.py)、[文字控制](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/scripts/selftrained/prepare_text_tools.py)和[評分彙總](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/scripts/selftrained/evaluate.py)，兩版原始數值見本節末的結果連結。

<details>
<summary>最後考卷另外檢查哪些工具邊界？</summary>

另外4類題目檢查工具規則，均使用固定語意判準：只問工具概念而不需執行，MoE為6／8、Dense為3／8；缺少必要參數而應詢問，為5／8與8／8；工具未開啟而應說明，為29／48與24／48；要求不支援的運算而應澄清，兩版均為4／8。這些題目和上表一起構成完整3734題，不等於又多測了一份考卷。

</details>

分開看入口與回答，還會發現一個有教學用的反例：兩版OCR入口的 CTC 讀字都完整辨對324／324，但共同語言核心生成的最終字串只有上表的252／324與281／324。[19.6的CTC入口](https://birdhackor.github.io/tiny-perceptron-vlm/19.6.html)另有逐欄預測合成字串的獨立檢查：這裡的324／324是那個字串與標準答案完整相同的題數。入口辨對，不保證後面的回答器會忠實保留字序、範圍與完整內容。因此評測要追到使用者真正收到的答案，不能只用中間辨識分數代表整個助理。

本表的能力範圍也由材料決定。OCR只含12個字，使用者提供一個連續1–4格的文字框；Sans與Serif兩種字體都在訓練中出現，不是自動找文字或陌生字體測試。圖片限三類服飾與已提供的兩格位置。語音來自同一來源內的有限客服意圖：最後保留的是30份錄音，衍生出不同歷史與要求，並非30位新說話者；來源沒有 speaker ID，沒有驗收說話者隔離的泛化。新問法與未見組合的結果支持上述具體範圍，不能換成一般中文、任意照片或任意真人語音的宣稱。

原本的判準沒有因結果不理想而調低。例如文字要逐意圖達80%，OCR整串要達80%，工具真正往返要達95%；高分任務不能抵銷另一項未達標。兩版的真正工具往返都低於95%，因此**兩版都沒有通過全部能力驗收**。訓練、公開下載與最後評測完成，讓我們能查清成品；它們和能力通過是不同的結論。[完整結果](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/results/v2-final-public-results.json)、[MoE原始彙總](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/results/public-raw/moe/test/metrics.json)、[Dense原始彙總](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/results/public-raw/dense/test/metrics.json)與[檔案索引](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/results/public-raw/index.json)保留可回查的分項；[MoE判準明細](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/results/v2-moe-public-final-test-stage-index.json)與[Dense判準明細](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/results/v2-dense-public-final-test-stage-index.json)列出未達項目。

練習用人工分數看平均的差別：若一類12題全部答對，另一類6題全部答錯，按題合計與每類等權重會得到不同摘要。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
by_question = (12 + 0) / (12 + 6)
by_task = (1.0 + 0.0) / 2
print("按題合計", by_question, "每類等權重", by_task)

前者是三分之二，後者是一半；這些是假設分數，不是本模型成績。把同一種題多出幾份，會改變按題合計的比重，卻不會自動增加能力。因此交付時先列各項判準與分母，再選摘要，而不是讓總分遮住重要失敗。

這個有限成品讓我們走完從隨機權重、分階段學習到同一核心實際回答的主線，並知道能力在哪裡中斷。接下來的[第20章](https://birdhackor.github.io/tiny-perceptron-vlm/chapter-20.html)換一個起點：接手成熟 Qwen 圖文權重，由 Whisper 處理語音轉寫，再研究應用與驗收。它是成熟模型的延伸，不是本章 MoE 的接續訓練，也不替這份成品補上能力分數。
